## A character-level implementation of [Bengio et al. 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)

In their original paper they associate each of 17,000 <span style="color:red">words</span> with a 30-dimensional vector (called feature vector).

This is a concept of (vector) embedding of words.

这些 associations 被存放在了一个 lookup table 里：一个 17000 * 30 的矩阵。

In the beginning, all the points (i.e., the feature vectors of the words) in this 30-dimensional space are initialized randomly, so they are spread out into all directions in this space. Then using backpropagation those feature vectors are tuned So that words with similar meaning move close together, and words that are semantically different are pointing in different directions in this (semantic) space.

They <span style="color:red">maximize the log likelihood</span> to train the multi-layer neural network.

During test time, prefix could be a sequence that never appears in the training set. 但 prefix 中的很多 tokens 在 semantic space 里和 neural network 见过的很多 tokens 离得很近。<span style="color:red">Neural network 大概知道这些它见过的 tokens 后面的 prediction 是什么</span>，所以 neural network 就能对这个没见过的 sequence 给出一个相似的 prediction。

In [113]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [114]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [115]:
len(words)

32033

### Vocabulary & Mappings to/from indices

In [116]:
letters = sorted(list(set(''.join(words))))
str_to_idx = {s:i+1 for i, s in enumerate(letters)}
str_to_idx['.'] = 0
idx_to_str = {i:s for s, i in str_to_idx.items()}

print(idx_to_str)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


### Build the Dataset

In [117]:
block_size = 3  # size of the prefix context
X = []  # inputs
Y = []  # labels

for w in words[:2]:
    print(w)
    context = [0] * block_size  # [0,0,...,0]
    # Note here that we use the special token '.' for padding
    for ch in w + '.':
        index = str_to_idx[ch]
        X.append(context)
        Y.append(index)

        print(''.join(idx_to_str[i] for i in context), '--->', idx_to_str[index])

        context = context[1:] + [index]   # roll the window

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .


In [118]:
X.shape, X.dtype

(torch.Size([12, 3]), torch.int64)

In [119]:
Y.shape, Y.dtype

(torch.Size([12]), torch.int64)

原论文有17,000个词汇，而我们有27个词汇（26 个英文字母 + 一个 special token）。

原论文用了30个维度去 embed 那17,000个词汇。

<span style="color:red">我们只需要用2个维度去 embed 这27个词汇。</span>

In [120]:
M = torch.randn((27, 2))
M


tensor([[-0.0449, -0.1414],
        [-0.3614,  0.2414],
        [ 1.6177, -0.5892],
        [-1.3574,  0.1553],
        [ 1.0940, -0.1368],
        [ 1.6162,  0.4893],
        [-1.0519,  0.9813],
        [ 0.3526, -0.4556],
        [ 0.9947,  0.7742],
        [ 0.7101,  0.6553],
        [ 0.0718,  1.0429],
        [-0.1195, -2.3744],
        [ 0.6900,  0.7147],
        [ 0.4333, -1.2191],
        [-1.5959, -1.1846],
        [ 0.4700,  0.8050],
        [ 0.8903,  0.0200],
        [ 1.2247, -1.2663],
        [ 0.4819,  1.9367],
        [ 0.5820, -0.2110],
        [-0.9169, -1.1825],
        [-1.6202,  0.7989],
        [-0.0121,  2.0272],
        [-0.1371, -0.1290],
        [-0.3781, -0.1404],
        [-1.5319, -0.4515],
        [-0.1972, -0.2657]])

我们可以用 one-hot vector 去提取某个词的 vector embedding:

In [121]:
M.dtype

torch.float32

In [122]:
F.one_hot(torch.tensor(5), num_classes=27).dtype

torch.int64

In [123]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ M

tensor([1.6162, 0.4893])

这里我们可以把“从embedding matrix M中提取token对应的embedded vector”理解成“用one hot encoding去表示输入tokens，然后把这些encoding（拼成一个input）feed进一个bias为 0 的linear layer里（这个layer中有embedding dimension * block_size个神经元，每个神经元有 vocabulary size * block_size 个weights），这个 linear layer 的输出就是这些 tokens 的 embedded vectors”

我们当然可以直接从embedding matrix M中取：

In [124]:
M[5]

tensor([1.6162, 0.4893])

We can also do:

In [125]:
M[torch.tensor([5,5,5])]

tensor([[1.6162, 0.4893],
        [1.6162, 0.4893],
        [1.6162, 0.4893]])

Recall:

In [126]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1]])

If we do:

In [127]:
M[X]

tensor([[[-0.0449, -0.1414],
         [-0.0449, -0.1414],
         [-0.0449, -0.1414]],

        [[-0.0449, -0.1414],
         [-0.0449, -0.1414],
         [ 1.6162,  0.4893]],

        [[-0.0449, -0.1414],
         [ 1.6162,  0.4893],
         [ 0.4333, -1.2191]],

        [[ 1.6162,  0.4893],
         [ 0.4333, -1.2191],
         [ 0.4333, -1.2191]],

        [[ 0.4333, -1.2191],
         [ 0.4333, -1.2191],
         [-0.3614,  0.2414]],

        [[-0.0449, -0.1414],
         [-0.0449, -0.1414],
         [-0.0449, -0.1414]],

        [[-0.0449, -0.1414],
         [-0.0449, -0.1414],
         [ 0.4700,  0.8050]],

        [[-0.0449, -0.1414],
         [ 0.4700,  0.8050],
         [ 0.6900,  0.7147]],

        [[ 0.4700,  0.8050],
         [ 0.6900,  0.7147],
         [ 0.7101,  0.6553]],

        [[ 0.6900,  0.7147],
         [ 0.7101,  0.6553],
         [-0.0121,  2.0272]],

        [[ 0.7101,  0.6553],
         [-0.0121,  2.0272],
         [ 0.7101,  0.6553]],

        [[-0.0121,  2

In [128]:
M[X].shape

torch.Size([12, 3, 2])

意为：对X每个元素取M对应的行

例如：

In [129]:
M[X][1,2]

tensor([1.6162, 0.4893])

In [130]:
X[1,2]

tensor(5)

In [131]:
M[5]

tensor([1.6162, 0.4893])

因此，`M[X]` embed 了 `X` 中所有的 token

In [132]:
emb_X = M[X]

In [133]:
emb_X[:,0,:]   # 所有 training examples 的 prefix context 中第一个 token 的 embedded vector

tensor([[-0.0449, -0.1414],
        [-0.0449, -0.1414],
        [-0.0449, -0.1414],
        [ 1.6162,  0.4893],
        [ 0.4333, -1.2191],
        [-0.0449, -0.1414],
        [-0.0449, -0.1414],
        [-0.0449, -0.1414],
        [ 0.4700,  0.8050],
        [ 0.6900,  0.7147],
        [ 0.7101,  0.6553],
        [-0.0121,  2.0272]])

In [134]:
emb_X[:,0,:].shape

torch.Size([12, 2])

我们可以把input拼起来：

In [135]:
torch.cat([emb_X[:,0,:], emb_X[:,1,:], emb_X[:,2,:]], dim=1)

tensor([[-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  1.6162,  0.4893],
        [-0.0449, -0.1414,  1.6162,  0.4893,  0.4333, -1.2191],
        [ 1.6162,  0.4893,  0.4333, -1.2191,  0.4333, -1.2191],
        [ 0.4333, -1.2191,  0.4333, -1.2191, -0.3614,  0.2414],
        [-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  0.4700,  0.8050],
        [-0.0449, -0.1414,  0.4700,  0.8050,  0.6900,  0.7147],
        [ 0.4700,  0.8050,  0.6900,  0.7147,  0.7101,  0.6553],
        [ 0.6900,  0.7147,  0.7101,  0.6553, -0.0121,  2.0272],
        [ 0.7101,  0.6553, -0.0121,  2.0272,  0.7101,  0.6553],
        [-0.0121,  2.0272,  0.7101,  0.6553, -0.3614,  0.2414]])

An equivalent but scalable way to do this:

In [136]:
torch.cat(torch.unbind(emb_X, dim=1), dim=1)

tensor([[-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  1.6162,  0.4893],
        [-0.0449, -0.1414,  1.6162,  0.4893,  0.4333, -1.2191],
        [ 1.6162,  0.4893,  0.4333, -1.2191,  0.4333, -1.2191],
        [ 0.4333, -1.2191,  0.4333, -1.2191, -0.3614,  0.2414],
        [-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  0.4700,  0.8050],
        [-0.0449, -0.1414,  0.4700,  0.8050,  0.6900,  0.7147],
        [ 0.4700,  0.8050,  0.6900,  0.7147,  0.7101,  0.6553],
        [ 0.6900,  0.7147,  0.7101,  0.6553, -0.0121,  2.0272],
        [ 0.7101,  0.6553, -0.0121,  2.0272,  0.7101,  0.6553],
        [-0.0121,  2.0272,  0.7101,  0.6553, -0.3614,  0.2414]])

`torch.unbind(emb, dim)` 把 `emb` 沿其第dim个维度拆开铺成一个tuple。

Pytorch tensor在内存上永远是一维数列，但 there are extra layer of logic on how the data in the tensor can be retrieved:

In [137]:
a = torch.arange(18)
print(a)
print(a.shape)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])
torch.Size([18])


In [138]:
a.view(3,3,2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

`view` does not change the underlying storage of the tensor.

In [139]:
b = a.view(2,9)
print(b)
print(b.shape)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])
torch.Size([2, 9])


In [140]:
b.storage()

 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

More on this: see [PyTorch internals](https://blog.ezyang.com/2019/05/pytorch-internals/)

我们可以用 `view` 去高效地把input拼出来：

In [141]:
emb_X.shape

torch.Size([12, 3, 2])

In [142]:
emb_X.view(emb_X.shape[0],6)

tensor([[-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  1.6162,  0.4893],
        [-0.0449, -0.1414,  1.6162,  0.4893,  0.4333, -1.2191],
        [ 1.6162,  0.4893,  0.4333, -1.2191,  0.4333, -1.2191],
        [ 0.4333, -1.2191,  0.4333, -1.2191, -0.3614,  0.2414],
        [-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  0.4700,  0.8050],
        [-0.0449, -0.1414,  0.4700,  0.8050,  0.6900,  0.7147],
        [ 0.4700,  0.8050,  0.6900,  0.7147,  0.7101,  0.6553],
        [ 0.6900,  0.7147,  0.7101,  0.6553, -0.0121,  2.0272],
        [ 0.7101,  0.6553, -0.0121,  2.0272,  0.7101,  0.6553],
        [-0.0121,  2.0272,  0.7101,  0.6553, -0.3614,  0.2414]])

Equivalently:

In [143]:
emb_X.view(-1,6)   # -1 == lets pytorch infer

tensor([[-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  1.6162,  0.4893],
        [-0.0449, -0.1414,  1.6162,  0.4893,  0.4333, -1.2191],
        [ 1.6162,  0.4893,  0.4333, -1.2191,  0.4333, -1.2191],
        [ 0.4333, -1.2191,  0.4333, -1.2191, -0.3614,  0.2414],
        [-0.0449, -0.1414, -0.0449, -0.1414, -0.0449, -0.1414],
        [-0.0449, -0.1414, -0.0449, -0.1414,  0.4700,  0.8050],
        [-0.0449, -0.1414,  0.4700,  0.8050,  0.6900,  0.7147],
        [ 0.4700,  0.8050,  0.6900,  0.7147,  0.7101,  0.6553],
        [ 0.6900,  0.7147,  0.7101,  0.6553, -0.0121,  2.0272],
        [ 0.7101,  0.6553, -0.0121,  2.0272,  0.7101,  0.6553],
        [-0.0121,  2.0272,  0.7101,  0.6553, -0.3614,  0.2414]])

### Input Layer

In [158]:
inputs = emb_X.view(-1,6)

### Hidden Layer

In [149]:
num_neuron_hidden = 100   # number of neurons in the hidden layer
vector_embedding_dim = 2
num_prefix_context_token = 3

num_input = vector_embedding_dim * num_prefix_context_token

In [150]:
W_hidden = torch.randn((num_input, num_neuron_hidden))
b_hidden = torch.randn(100)

In [155]:
hidden_out = torch.tanh(inputs @ W_hidden + b_hidden)   # Note that b1 is broadcasted here
hidden_out

tensor([[ 0.0129,  0.2707,  0.1871,  ..., -0.9240,  0.4927, -0.9436],
        [ 0.3787,  0.9955, -0.8199,  ..., -0.9773, -0.1536, -0.9901],
        [-0.9972, -0.9966,  0.9433,  ..., -0.9860,  0.9908, -0.9729],
        ...,
        [-0.8124,  0.9968, -0.9955,  ..., -0.3497,  0.9482, -0.9976],
        [-0.9971,  0.9996, -0.9015,  ..., -0.9274, -0.8822, -0.9996],
        [-0.9939,  0.9961, -0.9467,  ..., -0.6367, -0.1181, -0.9999]])

### Output Layer

In [156]:
W_output = torch.randn((num_neuron_hidden, 27))
b_output = torch.randn(27)

In [163]:
logits = hidden_out @ W_output + b_output
logits

tensor([[-1.2560e+00, -1.5968e+01, -6.7170e-02, -1.1814e+01, -3.8227e+00,
          5.4172e+00, -2.0538e+00,  3.2099e+00,  5.6587e+00, -6.5531e-01,
         -8.0675e+00,  6.7757e+00, -7.3418e+00, -9.6671e+00,  2.6065e+00,
          1.5580e+00, -1.4852e+00, -3.0392e-01,  6.6313e+00, -8.0594e+00,
          5.3023e+00, -1.2629e-01,  1.7218e+00, -5.3202e+00,  7.2884e+00,
         -7.5814e-01,  1.3768e+00],
        [-1.4941e-01, -1.3961e+01,  7.4122e-01, -6.0735e+00,  1.3973e+00,
          1.9224e+00, -8.2857e+00,  2.5045e+00, -8.2745e-01, -1.1072e+01,
         -2.8402e+00,  1.1875e+01, -9.3405e+00, -6.4646e+00, -4.6218e+00,
          1.7571e+00,  8.2487e-01,  1.3552e+01,  5.8773e+00, -8.2044e+00,
          1.9536e+00,  1.1145e+01,  5.8946e+00, -3.1874e+00, -6.9449e+00,
          9.3810e+00,  1.2925e+00],
        [-4.9310e+00, -1.3634e+01, -7.2801e+00, -2.7792e+00, -8.4999e+00,
          1.2416e+01, -8.2560e+00, -5.6377e+00, -2.1239e+00, -7.1669e+00,
         -9.7644e+00,  1.4155e+01, -9.98

### Softmax

In [164]:
counts = logits.exp()
counts

tensor([[2.8478e-01, 1.1620e-07, 9.3504e-01, 7.4020e-06, 2.1868e-02, 2.2524e+02,
         1.2825e-01, 2.4778e+01, 2.8677e+02, 5.1928e-01, 3.1357e-04, 8.7628e+02,
         6.4788e-04, 6.3333e-05, 1.3552e+01, 4.7492e+00, 2.2645e-01, 7.3792e-01,
         7.5850e+02, 3.1610e-04, 2.0081e+02, 8.8136e-01, 5.5948e+00, 4.8919e-03,
         1.4633e+03, 4.6854e-01, 3.9624e+00],
        [8.6122e-01, 8.6470e-07, 2.0985e+00, 2.3030e-03, 4.0443e+00, 6.8372e+00,
         2.5210e-04, 1.2238e+01, 4.3716e-01, 1.5540e-05, 5.8416e-02, 1.4359e+05,
         8.7793e-05, 1.5577e-03, 9.8353e-03, 5.7958e+00, 2.2816e+00, 7.6798e+05,
         3.5683e+02, 2.7345e-04, 7.0537e+00, 6.9200e+04, 3.6309e+02, 4.1280e-02,
         9.6352e-04, 1.1861e+04, 3.6419e+00],
        [7.2191e-03, 1.1992e-06, 6.8913e-04, 6.2090e-02, 2.0349e-04, 2.4678e+05,
         2.5970e-04, 3.5610e-03, 1.1957e-01, 7.7172e-04, 5.7459e-05, 1.4044e+06,
         4.6146e-05, 6.0621e-06, 6.2055e+01, 1.5263e+00, 1.2165e-02, 5.9821e+00,
         2.0256e+

In [165]:
prob = counts / counts.sum(1, keepdim=True)

沿 dim=1（27 个字符那个维度）求和，keepdim=True 让结果保持 (N, 1) 而不是 (N,)

(N, 27) / (N, 1) 触发 broadcasting：那一列和被横向复制 27 份，于是每一行除以自己那一行的总和

In [166]:
print(prob.shape)
print(prob[0].sum())

torch.Size([12, 27])
tensor(1.)


### NLL Loss

In [167]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0])

In [169]:
prob[torch.arange(prob.shape[0]), Y]

tensor([5.8236e-02, 1.5680e-09, 2.9897e-16, 9.1128e-13, 7.5650e-11, 1.2279e-03,
        6.8760e-06, 5.7535e-08, 1.9135e-13, 8.0932e-05, 1.9986e-05, 1.2016e-03])

In [168]:
prob[torch.arange(prob.shape[0]), Y].log()

tensor([ -2.8433, -20.2735, -35.7462, -27.7239, -23.3049,  -6.7024, -11.8875,
        -16.6709, -29.2847,  -9.4219, -10.8205,  -6.7241])

In [170]:
nll_loss = -(prob[torch.arange(prob.shape[0]), Y].log().mean())
nll_loss

tensor(16.7836)

Negative Log Likelihood is also called Cross Entropy:

In [171]:
F.cross_entropy(logits, Y)

tensor(16.7836)

`cross_entropy`'s implementation is much more efficient (due to that 1. for forward pass it uses <span style="color:red">fused kernel that clusters mathematical operations</span>, 2. for backward pass it is implemented with mathematical expressions that are simplifed analytically) and numerically stable (by offseting logits by their maximum, which does not change the probability computed. Note that if a logit is big during optimization, exp it will cause `inf` in program).

### Minibatch

For each step, instead of calculating the loss for the entire dataset (which is slow per step), we can calculate the loss for a small random subset (faster per step). By doing so, <span style="color:red">the gradient computed will be an estimation of the gradient for the full set</span>.

A small minibatch could be served as a good approximation of the full gradient, and thus using minibatches to optimize for more steps rather than using full batches to optimize for fewer steps yields better results.

### Determine a good learning rate

todo

### Full Network

In [202]:
# Full Dataset:

block_size = 3  # size of the prefix context
X = []  # inputs
Y = []  # labels

for w in words:
    context = [0] * block_size  # [0,0,...,0]
    # Note here that we use the special token '.' for padding
    for ch in w + '.':
        index = str_to_idx[ch]
        X.append(context)
        Y.append(index)
        context = context[1:] + [index]   # roll the window

X = torch.tensor(X)
Y = torch.tensor(Y)

X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [203]:
minibatch_size = 32
num_step = 10000
learning_rate = 0.1

M = torch.randn((27,2))
W_hidden = torch.randn((6,100))
b_hidden = torch.randn(100)
W_output = torch.randn((100, 27))
b_output = torch.randn(27)

parameters = [M, W_hidden, b_hidden, W_output, b_output]
for p in parameters:
    p.requires_grad = True

print('Total number of parameters in our MLP:', sum(p.nelement() for p in parameters))

Total number of parameters in our MLP: 3481


In [204]:
for _ in range(num_step):
    # minibatch construction:
    ix = torch.randint(0, X.shape[0], (minibatch_size,))

    # forward pass:
    emb_X = M[X[ix]]
    hidden_out = torch.tanh(emb_X.view(-1,6) @ W_hidden + b_hidden)
    logits = hidden_out @ W_output + b_output
    loss_for_minibatch = F.cross_entropy(logits, Y[ix])


    # backward pass:
    for p in parameters:
        p.grad = None   # each step needs to recalculate the gradients
    loss_for_minibatch.backward()   # calculate the gradients

    # parameters update:
    for p in parameters:
        p.data += -learning_rate * p.grad

print('Loss for the minibatch:', loss_for_minibatch.item())

emb_X = M[X]
hidden_out = torch.tanh(emb_X.view(-1,6) @ W_hidden + b_hidden)
logits = hidden_out @ W_output + b_output
loss_full = F.cross_entropy(logits, Y)
print('Loss for the full dataset:', loss_full.item())

Loss for the minibatch: 2.3749890327453613
Loss for the full dataset: 2.550381898880005


claude --resume abe4025c-d40a-4ab7-9825-d9de61d6adb0